**Liquid ranges of the elements.**
The **liquid range** of an element is the temperature interval between
its melting point and its boiling point - the range over which it exists
as a liquid at standard pressure.
Elements vary enormously: helium has the narrowest liquid range of any
element (only a few kelvin), while tungsten remains liquid across
thousands of degrees.

This notebook reads element data from `periodic_table.json`,
sorts the elements by group and period, computes each liquid range,
and plots the melting and boiling points together so the liquid ranges
are visible as the vertical gaps between the two curves.
Elements with unknown melting or boiling points are excluded from the
extremum search.

In [ ]:
"""plot_liquid_range.ipynb"""

# Cell 01 - Load the periodic table JSON and sort elements by group, period, number

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

with Path("periodic_table.json").open("r", encoding="utf-8") as infile:
    periodic_table = json.load(infile)

# Sort by group, then period, then atomic number
# (groups 0 are lanthanides/actinides; they sort first)
raw = sorted(
    periodic_table["elements"],
    key=lambda v: (int(v["group"]), int(v["period"]), int(v["number"])),
)

print(f"Elements loaded : {len(raw)}")
print(f"First entry     : {raw[0]['name']}  (group {raw[0]['group']}, period {raw[0]['period']})")

**Extracting temperatures.**
Melting and boiling point data is missing for some elements (stored as
`null` in JSON, which becomes `None` in Python).
These are replaced with sentinel values so NumPy arithmetic stays valid:
`-inf` for an unknown melt point and `+inf` for an unknown boil point.
This ensures the liquid range ($T_\text{boil} - T_\text{melt}$) for
any element with a missing value is $+\infty$, which naturally excludes
it from the minimum search and is caught by `np.isfinite` for the maximum.

In [ ]:
# Cell 02 - Build label, melt, and boil arrays; convert K to Celsius

labels = [f"{v['symbol']}-{v['number']}" for v in raw]

melt_k = np.array(
    [float(v["melt"]) if v["melt"] is not None else -np.inf for v in raw]
)
boil_k = np.array(
    [float(v["boil"]) if v["boil"] is not None else np.inf for v in raw]
)

# Convert from Kelvin to Celsius for display
melt = melt_k - 273.15
boil = boil_k - 273.15

# Liquid range: elements with any unknown value will have +inf range
liquid_range = boil - melt

n_known = np.sum(np.isfinite(liquid_range))
print(f"Elements with complete melt/boil data : {n_known} of {len(labels)}")

**Finding the extremes.**
The element with the smallest liquid range is found directly with
`np.argmin` - unknown values produce $+\infty$ ranges which are
never the minimum.
For the maximum, `np.isfinite` masks out elements with unknown data
before searching.

In [ ]:
# Cell 03 - Find elements with smallest and largest liquid range

finite_mask = np.isfinite(liquid_range)

min_idx = np.argmin(liquid_range)
max_idx = np.argmax(np.where(finite_mask, liquid_range, -np.inf))

print(
    f"Smallest liquid range : {liquid_range[min_idx]:>8,.2f} C  ->  {labels[min_idx]}"
)
print(
    f" Largest liquid range : {liquid_range[max_idx]:>8,.2f} C  ->  {labels[max_idx]}"
)

**Plotting melting and boiling points.**
Each element is placed at an integer x-position in the sort order
(group, period, atomic number).
The vertical distance between the turquoise (melt) and coral (boil)
curves at each element is its liquid range.
Elements with unknown data appear as gaps or extreme values in the curves.

In [ ]:
# Cell 04 - Plot melting and boiling points for all elements

x = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(18, 6), constrained_layout=True)
fig.canvas.manager.set_window_title("plot_liquid_range")

ax.plot(x, melt, color="turquoise", marker=".", label="Melting Point")
ax.plot(x, boil, color="coral",    marker=".", label="Boiling Point")

ax.set_title("Melting and Boiling Points of the Elements")
ax.set_xlabel("Elements (sorted by group, period, atomic number)")
ax.set_ylabel("Temperature (°C)")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=7, rotation=90)
ax.legend(loc="lower center")
ax.grid(True, alpha=0.3)

plt.show()